# Verifikasi kandidat SENA v0.8.3 bersama NARA

Memverifikasi ekspor melalui hash, mengulang prediksi pada set development yang sudah dilihat, dan menjalankan suite kontrak. Bukan uji penerimaan baru. NARA dan cutoff 0,95 tetap.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path(os.environ['SLR09_PACKAGE_ROOT']).resolve()
WORK = Path(os.environ['SLR09_WORK_DIR']).resolve()
TRAIN = Path(os.environ['SLR09_TRAINING_RUN']).resolve()
ARTIFACTS = Path(os.environ['SLR09_MODEL_OUTPUT']).resolve()
NARA = Path(os.environ['SLR09_NARA_ARCHIVE']).resolve()
def command(argv):
    proc = subprocess.Popen([sys.executable, *map(str, argv)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout: print(line, end='', flush=True)
    proc.wait()
    if proc.returncode: raise RuntimeError('Actual command failed: ' + str(argv[0]))
command([ROOT/'code/quality_fix/export_context_candidate_v083.py', '--run', TRAIN, '--output', ARTIFACTS, '--warm-start', os.environ['SLR09_WARM_START']])
lock = json.loads((ROOT/'config/quality_candidate_asset_lock_v0.8.3.json').read_text())
command([ROOT/'code/quality_fix/verify_context_candidate_v083.py', '--training-run', TRAIN,
    '--sena-archive', ARTIFACTS/lock['models']['sena']['archive_name'], '--nara-archive', NARA,
    '--output', WORK/'export_verification27'])
command([ROOT/'code/run_active_tests.py', '--evidence-dir', WORK/'active_tests27'])
state = json.loads((WORK/'export_verification27/run_state.json').read_text())
assert state['status'] == 'SUCCEEDED' and state['all_known_joint_regressions_met']
print('EXPORTED_CANDIDATE_VERIFIED_ON_SEEN_REFERENCES_NOT_ACCEPTANCE', flush=True)
